# Shared training code — Part 2: training loop and first B2 run (GPU)

**What this part adds:**
- `src/train/loop.py`: one run = AdamW (lr 1e-4, weight decay 1e-4), cosine schedule, mixed precision, cross-entropy weighted by 1/√frequency with `ignore_index = 19`; validation mIoU every 45 steps, the best weights are evaluated on the test patches and logged in `runs.csv`; checkpoint on the Drive every 225 steps and resume after a disconnection.
- `src/train/batches.py`: a batch order that depends only on (seed, step), and the training and validation patches kept on the GPU (no disk reading during training).
- `scripts/train_head.py` and the configs `configs/baselines/b1_mlp_t19.yaml`, `b2_unet_t19.yaml`.

**Budget:** 3,600 optimizer steps of 32 patches for every variant and every label fraction (= 80 epochs of the 1,455 training patches of fold I).

**Needs:** Parts 1–7 and the shared training code part 1. **Runtime:** GPU T4. **Time:** about 25 min.

**Testing a branch before merging it into `main`:** set `BRANCH` in step 1 to the name of the branch.

**Files needed in the repository** (new or changed in this part): `requirements.txt`, `src/train/data.py`, `src/train/batches.py`, `src/train/loop.py`, `scripts/train_head.py`, `configs/baselines/b1_mlp_t19.yaml`, `configs/baselines/b2_unet_t19.yaml`, `tests/test_train_data.py`, `tests/test_loop.py`, `notebooks/baselines/02_train_loop.ipynb`.

## 1. Runtime, Drive and configuration
**Check:** a GPU (Tesla T4) is printed.

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS, RUNS_CSV = DRIVE / "runs", DRIVE / "results" / "runs.csv"
CHECK_RUNS, CHECK_CSV = DRIVE / "runs_check", DRIVE / "results" / "runs_check.csv"  # short test runs only
OUT = DRIVE / "results" / "baselines"
B2 = "configs/baselines/b2_unet_t19.yaml"

## 2. Code and tests
**Check:** `32 passed`, and the commit line shows the expected branch head.

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy the data to the local disk
Only TESSERA 2019 is needed for B1 and B2.

**Check:** every line ends with `ok=True`.

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Check run: interrupted, then resumed
A short B2 run (450 steps) is stopped at step 225, as a disconnection would, then started again with the same command. It goes to `runs_check/` and `results/runs_check.csv`, not to the main `runs.csv`.

**Check:** the first command ends with `stopped at step 225`; the second one starts with `resumed from step 225`, prints the validation lines up to step 450 and a test mIoU. The line `data on cuda: … GB` gives the GPU memory used by the data (about 4.2 GB for TESSERA on the 1,937 training and validation patches).

In [ ]:
!python scripts/train_head.py --config {B2} --processed {PROC} --runs-dir {CHECK_RUNS} --runs-csv {CHECK_CSV} --steps 450 --stop-at 225 --rerun
!python scripts/train_head.py --config {B2} --processed {PROC} --runs-dir {CHECK_RUNS} --runs-csv {CHECK_CSV} --steps 450

## 5. First real run: B2 (TESSERA 2019 + light UNet 32–64–128), fold I, 100% labels, seed 0
3,600 steps, about 10 min. If the runtime disconnects, run steps 1–3 again, then this cell: the run continues from its last checkpoint.

**Check:** the run `base_unet-t19-w32_100_foldI_s0` is in `results/runs.csv`. The milestone G1 expects B2 at about 50 mIoU or more; the linear head on the same test patches gave 58.1.

In [ ]:
!python scripts/train_head.py --config {B2} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds 0

## 6. Validation curve and comparison with the linear head
**Check:** the figure is saved in `results/baselines/` and the table compares B2 with `lin_t19` (seed 0).

**How to read it:** the validation mIoU should rise and then flatten; a best step far before 3,600 with a falling curve afterwards means overfitting, a curve still rising at 3,600 means the budget is short.

In [ ]:
import matplotlib.pyplot as plt

rid = "base_unet-t19-w32_100_foldI_s0"
info = json.loads((RUNS / rid / "run.json").read_text())
h = pd.DataFrame(info["history"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(h.step, h.val_miou)
ax[0].axvline(info["best_step"], ls="--", c="grey", label=f"best: step {info['best_step']}")
ax[0].set(xlabel="step", ylabel="validation mIoU", title=rid)
ax[0].legend()
ax[1].plot(h.step, h.train_loss)
ax[1].set(xlabel="step", ylabel="training loss (weighted cross-entropy)")
fig.tight_layout()
OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / f"{rid}_curve.png", dpi=120)
plt.show()

runs = pd.read_csv(RUNS_CSV)
cols = ["run_id", "miou", "oa", "macro_f1", "miou_boundary", "miou_core", "n_params", "train_seconds"]
print(runs[runs.run_id.isin([rid, "lin_t19_100_foldI_s0"])][cols].round(4).to_string(index=False))
print(f"best validation mIoU {info['best_val_miou']:.4f} at step {info['best_step']} of {info['config']['steps']}")